# EE 467 – Phishing Detection: Exploratory Data Analysis

**Team:** Pravir Goosari, Akshath Rao, Rohan Sriram  
**Dataset:** UCI Phishing Websites (11,055 instances, 30 features)  
**Task:** Binary classification — phishing (−1) vs. legitimate (1)


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_dataset, FEATURE_NAMES, TARGET_NAME, get_feature_descriptions
from src.preprocessing import FEATURE_GROUPS, get_class_distribution

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Load the Dataset

In [ ]:
df = load_dataset()
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.dtypes.value_counts()

In [ ]:
# Check for missing values
print('Missing values:', df.isnull().sum().sum())
print('\nUnique values per feature:')
df[FEATURE_NAMES].nunique().to_frame('n_unique').T

## 2. Class Distribution

In [ ]:
dist = get_class_distribution(df[TARGET_NAME])
print(f"Phishing  : {dist['phishing_count']} ({dist['phishing_pct']:.1f}%)")
print(f"Legitimate: {dist['legitimate_count']} ({dist['legitimate_pct']:.1f}%)")

fig, ax = plt.subplots(figsize=(5, 4))
counts = df[TARGET_NAME].value_counts().rename({-1: 'Phishing', 1: 'Legitimate'})
counts.plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'], rot=0)
ax.set_title('Class Distribution')
ax.set_ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{p.get_height():,}', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom')
plt.tight_layout()
plt.show()

## 3. Feature Value Distributions

Each feature is encoded as {−1, 0, 1} where the meaning depends on the feature.

In [ ]:
# Show the value counts for all features
print('Feature value distributions (% of total):')
pd.set_option('display.max_columns', None)
pct = df[FEATURE_NAMES].apply(lambda c: c.value_counts(normalize=True).round(3))
pct.T

In [ ]:
# Heatmap of feature value frequencies
fig, ax = plt.subplots(figsize=(14, 10))
freq_df = df[FEATURE_NAMES].apply(lambda c: c.value_counts(normalize=True)).fillna(0)
# freq_df rows are unique values (-1, 0, 1), columns are features
sns.heatmap(freq_df, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Proportion'})
ax.set_title('Feature Value Frequency Heatmap')
ax.set_xlabel('Feature')
ax.set_ylabel('Value')
plt.tight_layout()
plt.show()

## 4. Feature–Label Correlation

In [ ]:
# Point-biserial correlation between each feature and the target
correlations = df[FEATURE_NAMES].corrwith(df[TARGET_NAME]).sort_values()

fig, ax = plt.subplots(figsize=(8, 10))
colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in correlations]
ax.barh(correlations.index, correlations.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Target (Result)')
ax.set_xlabel('Pearson r')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Feature Distributions by Class

In [ ]:
# Top 10 most correlated features
top_features = correlations.abs().nlargest(10).index.tolist()

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for ax, feat in zip(axes, top_features):
    for label, color, name in [(-1, '#e74c3c', 'Phishing'), (1, '#2ecc71', 'Legitimate')]:
        subset = df[df[TARGET_NAME] == label][feat]
        counts = subset.value_counts(normalize=True).sort_index()
        ax.bar(counts.index + (0.15 if label == 1 else -0.15), counts.values,
               width=0.3, color=color, label=name, alpha=0.8)
    ax.set_title(feat, fontsize=8)
    ax.set_xticks([-1, 0, 1])
    ax.legend(fontsize=6)

plt.suptitle('Top 10 Features by Class', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Feature Correlation Matrix (inter-feature)

In [ ]:
corr_matrix = df[FEATURE_NAMES].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', vmin=-1, vmax=1,
            center=0, annot=False, ax=ax, linewidths=0.3)
ax.set_title('Inter-Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 7. Feature Groups Analysis

In [ ]:
from src.preprocessing import FEATURE_GROUPS

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (group_name, features) in zip(axes, FEATURE_GROUPS.items()):
    group_corr = df[features].corrwith(df[TARGET_NAME]).sort_values()
    colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in group_corr]
    ax.barh(group_corr.index, group_corr.values, color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'{group_name} Features')
    ax.set_xlabel('Correlation with Target')
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Feature-Target Correlation by Feature Group', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Key EDA Takeaways

- The dataset is **nearly balanced** (~55% phishing, ~45% legitimate) — no need for aggressive class balancing.
- All features are ordinal integers in {−1, 0, 1}; **no missing values**.
- The most predictive features tend to be URL-based (IP address, subdomains) and security (SSL state, domain age).
- Several features are highly correlated with each other — tree models that handle multi-collinearity well (RF, GB) may outperform linear models.
- No feature has perfect separability, confirming the need for an ensemble approach.